# 01: Pressure-to-Deformation Calibration Model

## Overview
This notebook establishes the quantitative relationship between applied pneumatic pressure (mbar) and microfluidic chamber-wall separation (µm), effectively calibrating the PDMS device as a physical force sensor.

### Pipeline Stages
1. **Input:** Segmented binary mask images for 4 calibration datasets (Cal-1 to Cal-4), each containing 8 frames at pressures 0–2000 mbar.
2. **Pre-processing:** Morphological operations (closing + dilation) to fill gaps and reduce noise in segmented walls.
3. **Metrology:** Linear resampling of wall coordinates to equal point counts, followed by one-to-one Euclidean distance mapping (memory-efficient alternative to full pairwise cdist).
4. **Output:** Deformation (µm) and area change (%) vs pressure CSV; verification overlay plots.

**Pixel size:** 0.06587 µm/px | **Pressures:** 0, 200, 400, 600, 800, 1000, 1500, 2000 mbar

## 1. Imports & Path Configuration

In [ ]:
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import measure
from scipy.ndimage import binary_closing, binary_dilation
from skimage.morphology import disk
from natsort import natsorted  

# ── Paths ─────────────────────────────────────────────────────────────────
base_dir    = os.path.dirname(os.getcwd())
masks_root  = os.path.join(base_dir, "data", "processed", "masks", "calibration")
output_root = os.path.join(base_dir, "data", "results")
output_csv  = os.path.join(output_root, "Calibration_Deformation_Results.csv")
plot_folder = os.path.join(output_root, "wall_verification", "calibration")

os.makedirs(output_root,  exist_ok=True)
os.makedirs(plot_folder,  exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
pixel_size = 0.06587   # µm per pixel
num_points = 5000      # resampling resolution (higher = more accurate, more RAM)
pressures  = [0, 200, 400, 600, 800, 1000, 1500, 2000]  # mbar

# 4 calibration datasets
datasets = [f"Cal-{i}" for i in range(1, 5)]
print(f"Calibration datasets : {datasets}")
print(f"Pressure levels (mbar): {pressures}")

## 2. Helper Functions

In [ ]:
def resample_points(coords, num_points):
    """
    Linearly resample a wall's pixel coordinates to `num_points` equally spaced points.
    This equalises the point count of both walls, enabling efficient one-to-one
    distance mapping without a full O(n²) pairwise distance matrix.
    (Report Section 2.1.3)
    """
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    cum   = np.insert(np.cumsum(dists), 0, 0)
    tgt   = np.linspace(0, cum[-1], num_points)
    rx    = np.interp(tgt, cum, coords[:, 0])
    ry    = np.interp(tgt, cum, coords[:, 1])
    return np.stack([rx, ry], axis=1)


def preprocess_mask(image_array):
    """
    Apply morphological operations to improve segmentation mask quality.
    - binary_closing : fills small gaps and holes in detected walls
    - binary_dilation: further connects near-broken wall boundaries
    """
    struct = disk(3)
    closed  = binary_closing(image_array > 0, structure=struct)
    dilated = binary_dilation(closed, structure=struct)
    return dilated


print("Helper functions defined.")

## 3. Batch Processing — All Calibration Datasets

Each dataset (Cal-1 to Cal-4) is processed independently. For every pressure frame:
1. Mask is inverted (Fiji exports inverted masks).
2. Morphological operations applied.
3. Two largest labelled regions identified as left and right walls.
4. Both walls resampled to `num_points` and one-to-one distances computed.
5. Verification overlay saved to `wall_verification/calibration/`.

In [ ]:
all_results = []

for dataset_name in datasets:
    dataset_path = os.path.join(masks_root, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"WARNING: folder not found — {dataset_path}")
        continue

    image_files = natsorted([f for f in os.listdir(dataset_path) if f.endswith('.tif')])

    if len(image_files) != len(pressures):
        raise ValueError(f"{dataset_name}: expected {len(pressures)} images, found {len(image_files)}.")

    distances_um, std_devs_um = [], []

    for img_name, pressure in zip(image_files, pressures):
        img_path = os.path.join(dataset_path, img_name)

        # Load and invert
        img_array = 255 - np.array(Image.open(img_path))

        # Pre-process 
        processed = preprocess_mask(img_array)

        # Label and extract two walls
        labeled  = measure.label(processed, connectivity=2)
        regions  = measure.regionprops(labeled)
        if len(regions) < 2:
            raise ValueError(f"{img_name}: fewer than 2 wall regions detected.")

        # Sort by area descending, take two largest (the actual walls)
        regions = sorted(regions, key=lambda r: r.area, reverse=True)[:2]
        w1 = resample_points(np.array(regions[0].coords), num_points)  
        w2 = resample_points(np.array(regions[1].coords), num_points)

        dists = np.sqrt(np.sum((w1 - w2)**2, axis=1))
        avg_um = np.mean(dists) * pixel_size
        std_um = np.std(dists)  * pixel_size

        distances_um.append(avg_um)
        std_devs_um.append(std_um)

        # Verification plot
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.imshow(img_array, cmap='gray')
        ax.plot(w1[:, 1], w1[:, 0], 'r-', linewidth=0.8, label='Wall 1')
        ax.plot(w2[:, 1], w2[:, 0], 'b-', linewidth=0.8, label='Wall 2')
        ax.legend(fontsize=8)
        ax.set_title(f"Wall Resampling: {dataset_name} — {img_name}", fontsize=9)
        ax.axis('off')
        out_name = f"{dataset_name}_{img_name.replace('.tif','.png')}"
        fig.savefig(os.path.join(plot_folder, out_name), dpi=150, bbox_inches='tight')
        plt.close(fig)

    # Deformation relative to 0 mbar baseline
    d0          = distances_um[0]
    deformations = [d - d0 for d in distances_um]
    area_changes = [(defo / d0) * 100 for defo in deformations]

    for i, p in enumerate(pressures):
        all_results.append({
            "Dataset":              dataset_name,
            "Pressure_mbar":        p,
            "Distance_um":          distances_um[i],
            "Std_um":               std_devs_um[i],
            "Deformation_um":       deformations[i],
            "Area_Change_pct":      area_changes[i],
        })

    print(f"{dataset_name} processed — {len(image_files)} frames.")

import pandas as pd
df_cal = pd.DataFrame(all_results)
df_cal.to_csv(output_csv, index=False)
print(f"\nCalibration results saved to: {output_csv}")
df_cal.head(8)

## 4. Visualisation — Deformation & Area Change vs Pressure

In [ ]:
colors = {'Cal-1':'steelblue','Cal-2':'tomato','Cal-3':'seagreen','Cal-4':'darkorange'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ds in datasets:
    sub = df_cal[df_cal['Dataset'] == ds]
    if sub.empty: continue
    c = colors.get(ds, 'gray')
    axes[0].errorbar(sub['Pressure_mbar'], sub['Deformation_um'],
                     yerr=sub['Std_um'], fmt='o-', color=c, label=ds, capsize=3)
    axes[1].plot(sub['Pressure_mbar'], sub['Area_Change_pct'],
                 'o-', color=c, label=ds)

axes[0].set_xlabel("Applied Pressure (mbar)", fontsize=12)
axes[0].set_ylabel("Wall Deformation (µm)", fontsize=12)
axes[0].set_title("Wall Deformation vs Pressure", fontsize=12)
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_xlabel("Applied Pressure (mbar)", fontsize=12)
axes[1].set_ylabel("Change in Area (%)", fontsize=12)
axes[1].set_title("Area Change (%) vs Pressure", fontsize=12)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_root, 'calibration_curves.png'), dpi=300)
plt.show()